# Step 2 - Professional Preprocessing Pipeline

A leakage-free preprocessing pipeline built with `Pipeline`, `ColumnTransformer`, `OneHotEncoder`, and `StandardScaler`. Feature types are detected automatically; all fitted statistics come only from training data.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

def _find_root():
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / "src").is_dir() and (cand / "data").is_dir():
            return cand
    return p

ROOT = _find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src import config as C
from src import viz
viz.setup_style()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("Repository root:", ROOT)


Repository root: /Users/rauankaztaev/IdeaProjects/Draft/project


In [2]:
from src import data, preprocessing as pp
from src import utils

df = data.load_clean()
x_train, x_test, y_train, y_test = data.get_splits(df)
print("Train:", x_train.shape, " Test:", x_test.shape)
print("Train positive (Bad) rate:", round(float(y_train.mean()), 4))
print("Test  positive (Bad) rate:", round(float(y_test.mean()), 4))

Train: (6492, 5)  Test: (1623, 5)
Train positive (Bad) rate: 0.437
Test  positive (Bad) rate: 0.4368


## 2.1 Automatic feature-type detection

In [3]:
num, cat = pp.detect_feature_types(x_train)
print("Numeric features:", num)
print("Categorical features:", cat)

Numeric features: ['Temp', 'Humidity', 'Light', 'CO2']
Categorical features: ['Fruit']


## 2.2 Build and fit the ColumnTransformer (on training data only)

In [4]:
pre_scaled = pp.build_preprocessor(x_train, scale=True, ohe=True)
X_tr = pre_scaled.fit_transform(x_train)
X_te = pre_scaled.transform(x_test)
feat_names = pp.get_feature_names(pre_scaled)
print("Transformed train matrix:", X_tr.shape)
print("Output features:", feat_names)

scaler = pre_scaled.named_transformers_["num"].named_steps["scaler"]
report = pd.DataFrame({"feature": num, "train_mean": scaler.mean_.round(4),
                       "train_scale": scaler.scale_.round(4)})
display(report)
utils.save_table(report, "scaler_stats", caption="StandardScaler statistics learned from the training split.", label="tab:scaler")

Transformed train matrix: (6492, 8)
Output features: ['num__Temp', 'num__Humidity', 'num__Light', 'num__CO2', 'cat__Fruit_Banana', 'cat__Fruit_Orange', 'cat__Fruit_Pineapple', 'cat__Fruit_Tomato']


,feature,train_mean,train_scale
0,Temp,23.8182,1.2688
1,Humidity,93.2654,3.2271
2,Light,22.7938,42.9653
3,CO2,325.6677,60.0218


{'csv': PosixPath('/Users/rauankaztaev/IdeaProjects/Draft/project/tables/scaler_stats.csv'),
 'tex': PosixPath('/Users/rauankaztaev/IdeaProjects/Draft/project/tables/scaler_stats.tex')}

**Leakage control.** The scaler means/scales above are computed *only* on the training split; `transform` applies them unchanged to the test split. During cross-validation the same fitting is repeated inside each fold, so no test-fold statistic ever influences training. One-hot encoding uses `handle_unknown='ignore'` so unseen categories cannot break inference.

## 2.3 End-to-end pipeline sanity check

In [5]:
from src import models
pipe = models.build_pipeline("Logistic Regression",
                             models.get_estimators()["Logistic Regression"], x_train)
pipe.fit(x_train, y_train)
print("Pipeline steps:", [s[0] for s in pipe.steps])
print("Test accuracy (LogReg sanity):", round(pipe.score(x_test, y_test), 4))

split_tbl = pd.DataFrame({"partition": ["train", "test"],
                          "n_samples": [len(x_train), len(x_test)],
                          "n_bad": [int(y_train.sum()), int(y_test.sum())],
                          "n_good": [int((1-y_train).sum()), int((1-y_test).sum())]})
display(split_tbl)
utils.save_table(split_tbl, "split_sizes", caption="Stratified train/test split sizes.", label="tab:split")

Pipeline steps: ['preprocessor', 'model']
Test accuracy (LogReg sanity): 0.9094


,partition,n_samples,n_bad,n_good
0,train,6492,2837,3655
1,test,1623,709,914


{'csv': PosixPath('/Users/rauankaztaev/IdeaProjects/Draft/project/tables/split_sizes.csv'),
 'tex': PosixPath('/Users/rauankaztaev/IdeaProjects/Draft/project/tables/split_sizes.tex')}

The pipeline cleanly chains preprocessing and model, guaranteeing the identical, reproducible feature space is used by every model in the benchmark.